# Phase 2 - San Semantic Retrieval Research

Purpose:
- Build the missing Phase 2 research bridge between Hoang Phase 1 routing and Hoang Phase 3 grounded QA.
- Inspect how the aviation corpus is chunked, embedded, indexed, searched, scored, and passed to the LLM layer.
- Compare retrieval strategies: `bm25`, `semantic`, `hybrid`, `metadata_first`, and `hybrid_rrf`.

This notebook mirrors the app/runtime implementation in `aviation_rag/phase2_indexing.py` and `aviation_rag/phase2_retrieval_engine.py`.

## 1. Setup

The default architecture wants `sentence-transformers/all-MiniLM-L6-v2` for 384-dim dense embeddings. If the model is not cached or the machine cannot access HuggingFace, the runtime falls back explicitly to `tfidf_svd_faiss_fallback` and reports that in diagnostics.

For fast local research, set `USE_FAST_FALLBACK = True`. For final dense retrieval validation, set it to `False` and ensure the MiniLM model is available.

In [1]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    load_dotenv = None

if load_dotenv is not None:
    load_dotenv(PROJECT_ROOT / ".env")

os.environ.setdefault("LANGSMITH_TRACING", "false")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "false")
os.environ.setdefault("RETRIEVAL_MAX_DOCS", "500")
os.environ.setdefault("PHASE2_INDEX_DIR", str(PROJECT_ROOT / "artifacts" / "phase2_index_notebook"))

USE_FAST_FALLBACK = True
if USE_FAST_FALLBACK:
    os.environ["PHASE2_EMBEDDING_MODEL"] = "tfidf_svd_fallback"

from aviation_rag.config import Settings, ensure_artifact_dirs
from aviation_rag.phase1_hoang_intent_routing import Phase1HoangIntentRouting
from aviation_rag.phase2_indexing import build_corpus_chunks
from aviation_rag.phase2_san_faiss_retrieval import Phase2SanFaissRetrieval
from aviation_rag.phase2_metrics import evaluate_ranking
from aviation_rag.io_utils import write_jsonl

settings = Settings()
ensure_artifact_dirs(settings)

safe_settings = {
    "data_path": str(settings.data_path),
    "phase2_index_dir": str(settings.phase2_index_dir),
    "embedding_model": settings.phase2_embedding_model,
    "retrieval_max_docs": settings.retrieval_max_docs,
    "langsmith_tracing": settings.langsmith_tracing,
    "langsmith_api_key": "configured" if settings.langsmith_api_key else "not set",
}
safe_settings


{'data_path': 'C:\\Users\\DELL\\Desktop\\Vinh Hoang\\Master Program\\Xử lý ngôn ngữ tự nhiên\\Project\\data\\kaggle\\ASRS-clean-dataset-aviation-safety.csv',
 'phase2_index_dir': 'C:\\Users\\DELL\\Desktop\\Vinh Hoang\\Master Program\\Xử lý ngôn ngữ tự nhiên\\Project\\artifacts\\phase2_index_notebook',
 'embedding_model': 'tfidf_svd_fallback',
 'retrieval_max_docs': 500,
 'langsmith_tracing': 'true',
 'langsmith_api_key': 'configured'}

## 2. Corpus Chunking Inspection

Phase 2 starts from the shared aviation corpus. The runtime extracts text fields, normalizes aviation abbreviations, deduplicates repeated chunks, and attaches metadata such as airport, weather, component, problem type, and inferred document type.

In [2]:
chunks = build_corpus_chunks(settings)
print("chunk_count:", len(chunks))
print("first_chunk:")
print(json.dumps(chunks[0].__dict__, ensure_ascii=False, indent=2)[:1800])

chunk_count: 546
first_chunk:
{
  "doc_id": "645583",
  "chunk_id": "645583#0",
  "chunk_text": "a b757-200 was dispatched with deferred non airworthy fuselage skin scratches near the l static port in the rvsm critical area. soft straight hair lines on acft skin below l static port area suspected to be caused by other station's lavatory svcing truck safety fiberglas flag poles contacting acft when apching forward lavatory svcing panel (as per auditor). formal request for burnishing non airworthy of flt marks on fuselage skin not documented on non routine card of technician's non routine write-up form. foreman deferral burnishing carried out by following station zzz1. gpm section 17-03 has been reviewed for proper line station maint item deferment procs.",
  "lexical_text": "a b757 200 was dispatched with deferred non airworthy fuselage skin scratches near the l static port in the rvsm critical area soft straight hair lines on aircraft skin below l static port area suspected to be cause

## 3. Phase 1 Input Contract

Phase 1 sends Phase 2 a normalized object containing the query, predicted intent, expanded queries, rewritten query, and retrieval plan. Phase 2 does not guess user intent again; it respects this routing contract.

In [3]:
phase1 = Phase1HoangIntentRouting(settings)
query = "engine warning checklist after takeoff"
phase1_output = phase1.build_output(query, top_k=5)
print(phase1_output.model_dump_json(indent=2))

{
  "query_id": "q_e22cbcbe",
  "query_raw": "engine warning checklist after takeoff",
  "query_normalized": "engine warning checklist after takeoff",
  "intent": "Incident_Report",
  "intent_confidence": 0.42972178485072937,
  "intent_source": "ml",
  "expanded_queries": [
    "engine warning checklist after takeoff",
    "engine warning checklist after takeoff aviation incident report",
    "engine warning checklist after takeoff event narrative",
    "engine warning checklist after takeoff safety occurrence summary"
  ],
  "rewritten_query": "aviation incident narrative lookup for: engine warning checklist after takeoff",
  "retrieval_plan": {
    "strategy": "semantic",
    "fallback_strategy": "hybrid",
    "top_k": 5,
    "filters": {},
    "routing_reason": "Narrative incident queries benefit from semantic similarity over safety reports."
  },
  "created_at": "2026-06-06T06:13:01.895491"
}


## 4. Build Phase 2 Retrieval Engine

The retrieval engine supports:
- Dense semantic retrieval via MiniLM embeddings + L2 normalization + FAISS `IndexFlatIP`.
- BM25 lexical retrieval for exact technical/procedure terms.
- Metadata scoring/filtering for operating conditions and structured fields.
- Weighted hybrid fusion and Reciprocal Rank Fusion (`hybrid_rrf`).

In [4]:
retrieval = Phase2SanFaissRetrieval(settings)
print("available:", retrieval.available)
print("build_error:", retrieval.build_error)
print("index_info:")
print(json.dumps(retrieval.info.__dict__ if retrieval.info else {}, ensure_ascii=False, indent=2))

available: True
build_error: None
index_info:
{
  "retrieval_backend": "phase2_dense_bm25_hybrid",
  "embedding_model": "fallback:tfidf_svd_fallback",
  "embedding_backend": "tfidf_svd_faiss_fallback",
  "embedding_dim": 128,
  "faiss_index_type": "IndexFlatIP",
  "normalization": "L2",
  "chunk_count": 546,
  "bm25_enabled": true,
  "index_dir": "C:\\Users\\DELL\\Desktop\\Vinh Hoang\\Master Program\\Xử lý ngôn ngữ tự nhiên\\Project\\artifacts\\phase2_index_notebook",
  "fallback_reason": "TF-IDF/SVD fallback was explicitly requested by PHASE2_EMBEDDING_MODEL."
}


## 5. Run One Retrieval Strategy

Inspect returned documents, score decomposition, and diagnostics.

In [5]:
phase1_output = phase1.build_output(query, top_k=5, strategy="hybrid")
phase2_output = retrieval.retrieve(phase1_output)

print("diagnostics:")
print(json.dumps(phase2_output.retrieval_diagnostics, ensure_ascii=False, indent=2))

print("top docs:")
for rank, doc in enumerate(phase2_output.topk_docs, start=1):
    print(rank, doc.doc_id, doc.metadata.get("document_type"), doc.scores)
    print(doc.chunk_text[:350].replace("\n", " "))
    print("---")

diagnostics:
{
  "adapter_mode": "faiss_retrieval",
  "contract_owner": "Quang San",
  "retrieval_backend": "phase2_dense_bm25_hybrid",
  "embedding_model": "fallback:tfidf_svd_fallback",
  "embedding_backend": "tfidf_svd_faiss_fallback",
  "embedding_dim": 128,
  "faiss_index_type": "IndexFlatIP",
  "normalization": "L2",
  "chunk_count": 546,
  "corpus_size": 546,
  "bm25_enabled": true,
  "metadata_filter_applied": false,
  "fusion_method": "weighted_linear_fusion",
  "score_weights": {
    "semantic": 0.5,
    "bm25": 0.35,
    "metadata": 0.15
  },
  "strategy_requested": "hybrid",
  "fallback_strategy": "hybrid",
  "routing_reason": "Manual override requested. Original routing intent was Incident_Report.",
  "dataset_path": "C:\\Users\\DELL\\Desktop\\Vinh Hoang\\Master Program\\Xử lý ngôn ngữ tự nhiên\\Project\\data\\kaggle\\ASRS-clean-dataset-aviation-safety.csv",
  "index_dir": "C:\\Users\\DELL\\Desktop\\Vinh Hoang\\Master Program\\Xử lý ngôn ngữ tự nhiên\\Project\\artifacts\\p

## 6. Compare Methods

A good demo should not hide the algorithm choice. This cell compares the same query under each strategy so we can see which retrieval behavior is better for a given intent.

In [6]:
strategies = ["bm25", "semantic", "hybrid", "metadata_first", "hybrid_rrf"]
comparison_rows = []
for strategy in strategies:
    row = phase1.build_output(query, top_k=5, strategy=strategy)
    output = retrieval.retrieve(row)
    first_doc = output.topk_docs[0] if output.topk_docs else None
    comparison_rows.append({
        "strategy": strategy,
        "top_doc": first_doc.doc_id if first_doc else None,
        "top_type": first_doc.metadata.get("document_type") if first_doc else None,
        "top_final": first_doc.scores.get("final") if first_doc else None,
        "fusion": output.retrieval_diagnostics.get("fusion_method"),
        "latency_ms": output.retrieval_diagnostics.get("latency_ms"),
        "embedding_backend": output.retrieval_diagnostics.get("embedding_backend"),
    })

comparison_rows

[{'strategy': 'bm25',
  'top_doc': '644834',
  'top_type': 'incident_report',
  'top_final': 0.9871611595153809,
  'fusion': 'weighted_linear_fusion',
  'latency_ms': 10.576400003628805,
  'embedding_backend': 'tfidf_svd_faiss_fallback'},
 {'strategy': 'semantic',
  'top_doc': '645033',
  'top_type': 'procedure',
  'top_final': 1.0,
  'fusion': 'weighted_linear_fusion',
  'latency_ms': 11.456000007456169,
  'embedding_backend': 'tfidf_svd_faiss_fallback'},
 {'strategy': 'hybrid',
  'top_doc': '642781',
  'top_type': 'incident_report',
  'top_final': 0.9085226058959961,
  'fusion': 'weighted_linear_fusion',
  'latency_ms': 10.306300013326108,
  'embedding_backend': 'tfidf_svd_faiss_fallback'},
 {'strategy': 'metadata_first',
  'top_doc': '647474',
  'top_type': 'procedure',
  'top_final': 0.9544031023979187,
  'fusion': 'weighted_linear_fusion',
  'latency_ms': 11.657499999273568,
  'embedding_backend': 'tfidf_svd_faiss_fallback'},
 {'strategy': 'hybrid_rrf',
  'top_doc': '645033',
  't

## 7. Lightweight Retrieval Evaluation

Heuristic sanity check (term overlap) below is kept for quick debugging.

**Gold-set benchmark** (human-labeled relevant ASRS `event_id` lists) lives in `data/phase2_retrieval_gold_labels.jsonl` and is evaluated in the next section via `aviation_rag/phase2_retrieval_eval.py` (Recall@K, Precision@K, MRR).

In [7]:
benchmark_cases = [
    {"query": "engine warning checklist after takeoff", "strategy": "bm25", "expected_terms": ["engine", "warning", "checklist"]},
    {"query": "crosswind turbulence during final approach", "strategy": "metadata_first", "expected_terms": ["crosswind", "turbulence", "approach"]},
    {"query": "engine failure after takeoff with emergency return", "strategy": "semantic", "expected_terms": ["engine", "takeoff", "emergency"]},
]

def doc_is_relevant(doc, terms):
    text = " ".join([doc.chunk_text, " ".join(str(v) for v in doc.metadata.values())]).lower()
    return any(term.lower() in text for term in terms)

metric_rows = []
for case in benchmark_cases:
    row = phase1.build_output(case["query"], top_k=5, strategy=case["strategy"])
    output = retrieval.retrieve(row)
    retrieved_ids = [doc.doc_id for doc in output.topk_docs]
    relevant_ids = [doc.doc_id for doc in output.topk_docs if doc_is_relevant(doc, case["expected_terms"])]
    metric = evaluate_ranking(retrieved_ids, relevant_ids, k=5, latency_ms=output.retrieval_diagnostics.get("latency_ms", 0.0))
    metric_rows.append({"query": case["query"], "strategy": case["strategy"], **metric.__dict__})

metric_rows

[{'query': 'engine warning checklist after takeoff',
  'strategy': 'bm25',
  'precision_at_k': 1.0,
  'recall_at_k': 1.0,
  'mrr': 1.0,
  'latency_ms': 10.229800012893975},
 {'query': 'crosswind turbulence during final approach',
  'strategy': 'metadata_first',
  'precision_at_k': 0.8,
  'recall_at_k': 1.0,
  'mrr': 1.0,
  'latency_ms': 11.32839999627322},
 {'query': 'engine failure after takeoff with emergency return',
  'strategy': 'semantic',
  'precision_at_k': 0.6,
  'recall_at_k': 1.0,
  'mrr': 1.0,
  'latency_ms': 11.73719999496825}]

## 7b. Gold-set Retrieval Benchmark

Human/project-labeled relevant document IDs (`event_id`) for 8 demo queries aligned with Phase 1 gold-set. Metrics are computed with stratified retrieval runs per strategy.

In [8]:
import pandas as pd
from aviation_rag.phase2_retrieval_eval import (
    run_retrieval_gold_benchmark,
    save_retrieval_gold_report,
)

gold_report = run_retrieval_gold_benchmark(
    settings,
    strategies=["bm25", "semantic", "hybrid", "metadata_first", "hybrid_rrf"],
    phase1_agent=phase1,
    retrieval_engine=retrieval,
)
save_retrieval_gold_report(gold_report, settings.phase2_gold_report_path)

strategy_rows = []
for strategy, payload in (gold_report.get("strategies") or {}).items():
    summary = payload.get("summary") or {}
    strategy_rows.append(
        {
            "strategy": strategy,
            "pass_rate": payload.get("pass_rate"),
            "recall_at_k": summary.get("recall_at_k"),
            "precision_at_k": summary.get("precision_at_k"),
            "mrr": summary.get("mrr"),
            "latency_ms": summary.get("latency_ms"),
        }
    )

print("Gold labels:", settings.phase2_gold_labels_path)
print("Saved report:", settings.phase2_gold_report_path)
print("Overall summary:", gold_report.get("summary"))
pd.DataFrame(strategy_rows)

Gold labels: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\data\phase2_retrieval_gold_labels.jsonl
Saved report: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase2_retrieval_gold_report.json
Overall summary: {'precision_at_k': 0.0, 'recall_at_k': 0.0, 'mrr': 0.0, 'latency_ms': 11.9535}


,strategy,pass_rate,recall_at_k,precision_at_k,mrr,latency_ms
0,bm25,0.0,0.0,0.0,0.0,11.0388
1,semantic,0.0,0.0,0.0,0.0,11.5550
2,hybrid,0.0,0.0,0.0,0.0,10.6438
3,metadata_first,0.0,0.0,0.0,0.0,10.3725
4,hybrid_rrf,0.0,0.0,0.0,0.0,16.1575


## 8. Export Phase 2 Artifact

The artifact is the handoff from San Phase 2 to Hoang Phase 3. It contains `query_id`, `predicted_intent`, `topk_docs`, and `retrieval_diagnostics`.

In [9]:
artifact_path = settings.artifacts_dir / "phase2_san_retrieval_output.jsonl"
write_jsonl(artifact_path, [phase2_output])
print("wrote:", artifact_path)
print(artifact_path.read_text(encoding="utf-8")[:1200])

wrote: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase2_san_retrieval_output.jsonl
{"query_id": "q_97795287", "predicted_intent": "Incident_Report", "topk_docs": [{"doc_id": "642781", "chunk_id": "642781#0", "chunk_text": "a b737-500 at fl310 diverted due to #1 generator tripped off twice. red #1 eng fire handle light on; and #1 eng low hydraulic pressure warning. enrte zzz to zzz1; e of zzz2 eng #1 gen tripped off line. was able to reconnect gen and all was normal for about 15 min. eng #1 gen again tripped off line. was not able to reconnect gen second time. started apu and powered the bus. shortly after that; the light in the #1 fire handle came on without any other warning (no bell; no master warn). called maint to discuss the sit and determined without the other warnings or eng indications; was a faulty light. while still talking to maint; we also had a #1 eng hyd low pressure light come on. at that point we felt enough abnormal thing

## 9. Research Conclusion

Phase 2 is the evidence engine. It receives intent-aware routing from Phase 1 and returns ranked, scored, citation-ready context for Phase 3.

Recommended interpretation:
- `bm25`: best when exact checklist/component terms matter.
- `semantic`: best when the user describes an incident narratively.
- `metadata_first`: best when weather, airport, runway, operating condition, or structured fields matter.
- `hybrid`: robust default combining semantic + keyword + metadata.
- `hybrid_rrf`: rank-fusion alternative that is often stable when score scales differ.

## 10. Academic Retrieval Metrics and Fusion Formulas

Core formulas:

- Cosine similarity after L2 normalization: $cos(q,d)=\frac{q \cdot d}{\|q\|\|d\|}$
- BM25: $\sum_{t \in q} IDF(t)\frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+b\frac{|d|}{avgdl})}$
- Weighted hybrid score: $0.50S_{semantic}+0.35S_{BM25}+0.15S_{metadata}$
- RRF: $RRF(d)=\sum_{r \in R}\frac{1}{k+rank_r(d)}$

Section **7b** runs the **gold-set benchmark** (`data/phase2_retrieval_gold_labels.jsonl`) with human/project-labeled relevant `event_id` lists. The heuristic term-overlap block below is kept only as a quick sanity check.


In [10]:
import json
from statistics import mean
from aviation_rag.phase2_metrics import evaluate_ranking, summarize_metric_rows

metric_results = []
metric_debug_rows = []
for case in benchmark_cases:
    routed = phase1.build_output(case['query'], top_k=5, strategy=case['strategy'])
    retrieved = retrieval.retrieve(routed)
    docs = retrieved.topk_docs
    retrieved_ids = [doc.doc_id for doc in docs]
    expected_terms = [term.lower() for term in case.get('expected_terms', [])]
    pseudo_relevant_ids = [
        doc.doc_id
        for doc in docs
        if any(term in doc.chunk_text.lower() for term in expected_terms)
    ]
    metrics = evaluate_ranking(
        retrieved_ids,
        pseudo_relevant_ids,
        k=5,
        latency_ms=float(retrieved.retrieval_diagnostics.get('latency_ms', 0.0)),
    )
    metric_results.append(metrics)
    metric_debug_rows.append({
        'query': case['query'],
        'strategy': case['strategy'],
        'top_doc': retrieved_ids[0] if retrieved_ids else None,
        'term_hits_in_topk': len(pseudo_relevant_ids),
        'precision_at_5': round(metrics.precision_at_k, 4),
        'recall_at_5': round(metrics.recall_at_k, 4),
        'mrr': round(metrics.mrr, 4),
        'latency_ms': round(metrics.latency_ms, 2),
    })

summary = summarize_metric_rows(metric_results)
summary = {key: round(value, 4) for key, value in summary.items()}
print(json.dumps({'summary': summary, 'rows': metric_debug_rows}, indent=2, ensure_ascii=False))


{
  "summary": {
    "precision_at_k": 0.2667,
    "recall_at_k": 0.3333,
    "mrr": 0.3333,
    "latency_ms": 11.587
  },
  "rows": [
    {
      "query": "engine warning checklist after takeoff",
      "strategy": "bm25",
      "top_doc": "644834",
      "term_hits_in_topk": 4,
      "precision_at_5": 0.8,
      "recall_at_5": 1.0,
      "mrr": 1.0,
      "latency_ms": 12.63
    },
    {
      "query": "crosswind turbulence during final approach",
      "strategy": "metadata_first",
      "top_doc": "645270",
      "term_hits_in_topk": 0,
      "precision_at_5": 0.0,
      "recall_at_5": 0.0,
      "mrr": 0.0,
      "latency_ms": 10.19
    },
    {
      "query": "engine failure after takeoff with emergency return",
      "strategy": "semantic",
      "top_doc": "642474",
      "term_hits_in_topk": 0,
      "precision_at_5": 0.0,
      "recall_at_5": 0.0,
      "mrr": 0.0,
      "latency_ms": 11.94
    }
  ]
}


## 11. Research Interpretation

Interpret the benchmark with care:

- High Precision@5 means the top retrieved ASRS chunks contain query-relevant terms or concepts.
- High MRR means an answer-bearing document appears early in the ranked list.
- Low latency means the app can support interactive exploration.
- If hybrid and RRF do not beat semantic/BM25 for every query, that is expected: the research question becomes when each routing strategy is best.

Recommended next step: create a manually labeled evaluation set with query, intent, relevant doc IDs, and answer support labels.
